# %% [markdown]
# # Pipeline EdNet KT1: làm sạch -> bộ dữ liệu -> CTT -> Rasch -> CTT đầy đủ (dữ liệu giả lập)
# Cách chạy: mở file trong VS Code, chạy từng ô "# %%" (Shift+Enter) hoặc chạy cả file bằng `python ednet_pipeline.py`.
# Mọi tham số nằm ở ô 0. Chạy thử với N_FILES nhỏ trước, xem số liệu đếm, rồi mới tăng lên.

In [21]:
# %% 0. Cấu hình
import os
import json
from pathlib import Path
from collections import Counter
 
import numpy as np
import pandas as pd
 
try:
    from tqdm.auto import tqdm
except ImportError:                      # không có tqdm thì vẫn chạy, chỉ không có thanh tiến trình
    def tqdm(x, **kwargs):
        return x
 
ROOT = Path(os.environ.get("EDNET_ROOT", r"C:\Datasets\EdNet"))
OUT = Path(os.environ.get("EDNET_OUT", r"C:\Datasets\processed"))
OUT.mkdir(parents=True, exist_ok=True)
 
N_FILES = int(os.environ.get("N_FILES", 5000))        # số học viên đọc (tăng dần khi đã ổn)
SEED = 42

In [22]:
 #--- ngưỡng làm sạch (lựa chọn của nhóm, không phải chuẩn của EdNet: hãy ghi vào báo cáo) ---
VALID = {"a", "b", "c", "d"}          # đáp án hợp lệ
MIN_RESP = 20                         # người có ít hơn số lượt này bị loại
MIN_STRAIGHT, TOP_SHARE = 30, 0.90    # chọn cùng một đáp án > 90% lượt (với >= 30 lượt) -> loại
MAX_ELAPSED = 600_000                 # ms; chỉ ĐÁNH DẤU, không xóa (elapsed_time tính cho cả nhóm câu)
 
# --- tham số khối dữ liệu cho CTT/IRT ---
N_ITEMS = int(os.environ.get("N_ITEMS", 200))         # số câu nhiều người làm nhất
MIN_ANS = int(os.environ.get("MIN_ANS", 50))          # người phải làm >= số câu này trong khối
MIN_ITEM_N = int(os.environ.get("MIN_ITEM_N", 100))   # câu cần >= số người này mới đáng tin

P_LOW, P_HIGH, R_MIN = 0.3, 0.7, 0.2

In [23]:
# %% 1. Câu hỏi: làm sạch và dựng Q-matrix
def clean_questions(q):
    q = q.copy()
    n0 = len(q)
    q["question_id"] = q["question_id"].astype(str).str.strip()
    q = q.drop_duplicates("question_id")
    q["correct_answer"] = q["correct_answer"].astype(str).str.strip().str.lower()
    q = q[q["correct_answer"].isin(VALID)]
    q["part"] = pd.to_numeric(q["part"], errors="coerce")
    # nhãn kỹ năng: giá trị thiếu thật sự -> danh sách rỗng (không để thành chuỗi "nan")
    q["tag_list"] = q["tags"].apply(
        lambda s: [t.strip() for t in str(s).split(";") if t.strip()] if pd.notna(s) else [])
    print(f"Câu hỏi: {n0} -> {len(q)} | không có nhãn kỹ năng: {(q['tag_list'].str.len() == 0).sum()}")
    return q
 
 
def build_qmatrix(q):
    e = (q[["question_id", "tag_list"]].explode("tag_list")
         .dropna(subset=["tag_list"]).reset_index(drop=True))          # explode lặp chỉ số -> phải đặt lại
    qm = pd.crosstab(e["question_id"], e["tag_list"]).clip(upper=1)
    return qm.reindex(q["question_id"], fill_value=0)
 
 
q = clean_questions(pd.read_csv(ROOT / "contents" / "questions.csv"))
q_matrix = build_qmatrix(q)
ans_map = q.set_index("question_id")["correct_answer"]
part_map = q.set_index("question_id")["part"]
print("Q-matrix:", q_matrix.shape)
print("Nhãn bất thường (không phải số):", [c for c in q_matrix.columns if not str(c).lstrip("-").isdigit()])
q_matrix.to_parquet(OUT / "q_matrix.parquet")
print(q["part"].value_counts().sort_index())

Câu hỏi: 13169 -> 13169 | không có nhãn kỹ năng: 0
Q-matrix: (13169, 189)
Nhãn bất thường (không phải số): []
part
1     643
2    1662
3    1266
4    1158
5    5703
6    1335
7    1402
Name: count, dtype: int64


In [24]:
# %% 2. Làm sạch từng học viên (lưu theo mảnh, có bảng đếm số dòng bị loại)
stats = Counter()
 
 
def clean_user(path):
    d = pd.read_csv(path, usecols=["timestamp", "solving_id", "question_id", "user_answer", "elapsed_time"])
    stats["1_dong_goc"] += len(d)
 
    d = d.dropna(subset=["timestamp", "question_id", "user_answer"])
    stats["2_sau_bo_thieu"] += len(d)
 
    d["user_answer"] = d["user_answer"].astype(str).str.strip().str.lower()
    d = d[d["user_answer"].isin(VALID)]
    stats["3_dap_an_hop_le"] += len(d)
 
    d = d[d["question_id"].isin(ans_map.index)]
    stats["4_cau_co_trong_bang_cau_hoi"] += len(d)
 
    d = d.drop_duplicates(["timestamp", "question_id"])
    stats["5_sau_bo_trung"] += len(d)
 
    if len(d) < MIN_RESP:
        stats["nguoi_bi_loai_qua_it_luot"] += 1
        return None
    if len(d) >= MIN_STRAIGHT and d["user_answer"].value_counts(normalize=True).iloc[0] > TOP_SHARE:
        stats["nguoi_bi_loai_chon_mot_dap_an"] += 1
        return None
 
    d = d.sort_values(["timestamp", "solving_id"], kind="mergesort")      # thứ tự thời gian (cần cho BKT)
    d["attempt_no"] = d.groupby("question_id").cumcount() + 1             # lần làm thứ mấy của câu đó
    d["correct"] = (d["user_answer"] == d["question_id"].map(ans_map)).astype(int)
    d["bad_time"] = (d["elapsed_time"] <= 0) | (d["elapsed_time"] > MAX_ELAPSED)
    d["user_id"] = Path(path).stem
    stats["nguoi_giu_lai"] += 1
    return d
 
 
files = sorted(str(p) for p in (ROOT / "KT1").glob("*.csv"))
print("Số file KT1:", len(files))
rng = np.random.default_rng(SEED)
sample = rng.choice(files, size=min(N_FILES, len(files)), replace=False)
 
clean_dir = OUT / "clean_all"
clean_dir.mkdir(exist_ok=True)
for old in clean_dir.glob("*.parquet"):          # tránh lẫn kết quả của lần chạy trước
    old.unlink()
 
buf, k = [], 0
for f in tqdm(sample):
    d = clean_user(f)
    if d is not None:
        buf.append(d)
    if len(buf) >= 5000:
        pd.concat(buf).to_parquet(clean_dir / f"part_{k:03d}.parquet")
        buf, k = [], k + 1
if buf:
    pd.concat(buf).to_parquet(clean_dir / f"part_{k:03d}.parquet")
 
print("\n== Bảng đếm làm sạch ==")
for key_, v in stats.items():
    print(f"{key_:34s}{v:>14,}")
(OUT / "clean_report.json").write_text(json.dumps(dict(stats), indent=2), encoding="utf-8")

Số file KT1: 784309

== Bảng đếm làm sạch ==
1_dong_goc                               652,345
2_sau_bo_thieu                           652,131
3_dap_an_hop_le                          652,131
4_cau_co_trong_bang_cau_hoi              652,131
5_sau_bo_trung                           652,117
nguoi_giu_lai                              2,129
nguoi_bi_loai_qua_it_luot                  2,871


214

In [25]:
# %% 3. Kiểm tra sau làm sạch và tạo các bộ dữ liệu
if not list(clean_dir.glob("*.parquet")):
    raise SystemExit("Không còn người nào sau làm sạch: kiểm tra lại ngưỡng ở ô 0 và bảng đếm ở ô 2.")
 
clean = pd.read_parquet(clean_dir)
assert clean["user_answer"].isin(VALID).all()
assert clean[["user_id", "question_id", "timestamp", "correct"]].notna().all().all()
assert clean.groupby("user_id")["timestamp"].apply(lambda s: s.is_monotonic_increasing).all()
print(clean.shape, "| số người:", clean["user_id"].nunique(),
      "| tỷ lệ đúng:", round(clean["correct"].mean(), 3),
      "| tỷ lệ bad_time:", round(clean["bad_time"].mean(), 3))
 
clean.to_parquet(OUT / "ednet_all_attempts.parquet")                   # cho BKT: mọi lượt, theo thời gian
first = clean[clean["attempt_no"] == 1]
first.to_parquet(OUT / "ednet_first_attempts.parquet")                # cho CTT/IRT: lần đầu tiên
first5 = first[first["question_id"].map(part_map) == 5]               # Part 5: câu đứng độc lập
first5.to_parquet(OUT / "ednet_first_part5.parquet")
print("lần đầu:", first.shape, "| Part 5:", first5.shape)

(630406, 9) | số người: 2129 | tỷ lệ đúng: 0.665 | tỷ lệ bad_time: 0.003
lần đầu: (522452, 9) | Part 5: (207780, 9)


In [26]:
# %% 4. Khối dữ liệu dày cho CTT/IRT (Part 5, lần làm đầu tiên)
def build_block(df, n_items=N_ITEMS, min_ans=MIN_ANS):
    top = df["question_id"].value_counts().head(n_items).index
    b = df[df["question_id"].isin(top)]
    cnt = b["user_id"].value_counts()
    b = b[b["user_id"].isin(cnt[cnt >= min_ans].index)]
    return b.pivot(index="user_id", columns="question_id", values="correct")
 
 
R = build_block(first5)
print("R:", R.shape, "| tỷ lệ ô có dữ liệu:", round(R.notna().mean().mean(), 3))
print("số câu mỗi người (trung vị):", R.notna().sum(axis=1).median())
print("số người mỗi câu (nhỏ nhất / trung vị):", R.notna().sum().min(), "/", R.notna().sum().median())
R.to_parquet(OUT / "R_part5.parquet")

R: (275, 200) | tỷ lệ ô có dữ liệu: 0.498
số câu mỗi người (trung vị): 89.0
số người mỗi câu (nhỏ nhất / trung vị): 72 / 129.5


In [27]:
# %% 5. CTT trên ma trận thưa (EdNet): độ khó, tương quan câu-phần còn lại, phân loại theo bài báo
def classify(p, r):
    if P_LOW <= p <= P_HIGH and r > R_MIN:
        return "phu hop"
    issues = []
    if p < P_LOW:
        issues.append("qua kho")
    if p > P_HIGH:
        issues.append("qua de")
    if pd.isna(r) or r <= R_MIN:
        issues.append("phan cach thap")
    return ", ".join(issues)
 
 
def ctt_sparse(R, min_n=MIN_ITEM_N):
    n_ans, sum_ans = R.notna().sum(axis=1), R.sum(axis=1)
 
    def rest_corr(col):
        # điểm "phần còn lại" theo tỷ lệ đúng, vì mỗi người làm số câu khác nhau
        rest = (sum_ans - col.fillna(0)) / (n_ans - col.notna().astype(int))
        return col.corr(rest)
 
    t = pd.DataFrame({"n": R.notna().sum(), "p": R.mean(), "r_rest": R.apply(rest_corr)})
    t["class"] = [classify(p, r) for p, r in zip(t.p, t.r_rest)]
    t.loc[t.n < min_n, "class"] = "it du lieu (n < %d)" % min_n       # cờ này ưu tiên: kết quả chưa đáng tin
    return t
 
 
ctt = ctt_sparse(R)
print(ctt.describe().round(3))
print(ctt["class"].value_counts())
ctt.to_csv(OUT / "ctt_part5.csv")

             n        p   r_rest
count  200.000  200.000  200.000
mean   137.065    0.536    0.187
std     35.089    0.158    0.122
min     72.000    0.110   -0.147
25%    112.000    0.441    0.106
50%    129.500    0.535    0.194
75%    161.250    0.657    0.276
max    258.000    0.867    0.472
class
phu hop                    74
phan cach thap             67
it du lieu (n < 100)       20
qua de, phan cach thap     14
qua de                     13
qua kho, phan cach thap    11
qua kho                     1
Name: count, dtype: int64


In [28]:
# %% 6. Rasch thử nghiệm bằng girth (cần: pip install girth)
try:
    from girth import rasch_mml, tag_missing_data
except ImportError:
    print("Chưa cài girth: bỏ qua ô này (pip install girth).")
else:
    n_item, p_item = R.notna().sum(), R.mean()
    R2 = R.loc[:, (n_item >= MIN_ITEM_N) & (p_item > 0.02) & (p_item < 0.98)]
    n_per, s_per = R2.notna().sum(axis=1), R2.mean(axis=1)
    R2 = R2.loc[(n_per >= 20) & (s_per > 0) & (s_per < 1)]
    print("R2:", R2.shape)
    data = np.nan_to_num(R2.T.to_numpy(), nan=-1).astype(int)           # girth: (câu x người); -1 = thiếu
    res = rasch_mml(tag_missing_data(data, [0, 1]))
    print("Các khóa kết quả:", list(res.keys()))
    b = pd.Series(res["Difficulty"], index=R2.columns, name="b")
    cmp_ = pd.DataFrame({"b": b, "p": R2.mean()})
    print(cmp_.corr(method="spearman").round(3))                        # kỳ vọng: rất âm, gần -1
    cmp_.to_csv(OUT / "rasch_part5.csv")

R2: (275, 180)
Các khóa kết quả: ['Discrimination', 'Difficulty']
     b    p
b  1.0 -1.0
p -1.0  1.0


In [29]:
# %% 7. CTT đầy đủ cho một kỳ thi (mọi thí sinh làm cùng bộ câu): D27%, r_pbis, KR-20, phương án nhiễu
def score(A, key):
    return (A == key).astype(int)
 
 
def kr20(R):
    k, p = R.shape[1], R.mean()
    return (k / (k - 1)) * (1 - (p * (1 - p)).sum() / R.sum(axis=1).var(ddof=0))
 
 
def _groups(R, frac=0.27):
    total = R.sum(axis=1).sort_values(kind="mergesort")
    n = int(round(len(R) * frac))
    return total.index[:n], total.index[-n:]                            # nhóm thấp, nhóm cao
 
 
def d27(R):
    low, high = _groups(R)
    return R.loc[high].mean() - R.loc[low].mean()
 
 
def r_pbis(R, corrected=False):
    total = R.sum(axis=1)
    return R.apply(lambda c: c.corr(total - c if corrected else total))
 
 
def distractors(A, R, key, options=("A", "B", "C", "D"), min_share=0.05):
    low, high = _groups(R)
    rows = []
    for item in A.columns:
        for opt in options:
            pa = (A[item] == opt).mean()
            ph = (A.loc[high, item] == opt).mean()
            pl = (A.loc[low, item] == opt).mean()
            is_key = key[item] == opt
            flag = ""
            if is_key and ph <= pl:
                flag = "dap an dung khong phan biet duoc nhom gioi"
            if not is_key and pa < min_share:                           # min_share là quy ước, có thể chỉnh
                flag = "it nguoi chon"
            if not is_key and ph > pl:
                flag = "thu hut nhom gioi hon nhom yeu"
            rows.append(dict(item=item, option=opt, is_key=is_key, p_all=pa, p_high=ph, p_low=pl, flag=flag))
    return pd.DataFrame(rows)
 
 
def analyze(A, key, corrected=False):
    R = score(A, key)
    items = pd.DataFrame({"p": R.mean(), "d27": d27(R),
                          "r_pbis": r_pbis(R, corrected), "r_pbis_rest": r_pbis(R, True)})
    r = items["r_pbis_rest"] if corrected else items["r_pbis"]
    items["class"] = [classify(p, rr) for p, rr in zip(items.p, r)]
    return {"kr20": kr20(R), "items": items, "distractors": distractors(A, R, key)}
 
 
def simulate_exam(n_items=120, n_persons=2000, guess=0.2, seed=42, options="ABCD"):
    rng = np.random.default_rng(seed)
    theta = rng.normal(0, 1, n_persons)
    a = rng.lognormal(0, 0.3, n_items)                                  # độ phân cách thật
    b = rng.normal(0, 1, n_items)                                       # độ khó thật
    p = guess + (1 - guess) / (1 + np.exp(-a * (theta[:, None] - b)))
    correct = rng.random(p.shape) < p
    key = rng.choice(list(options), n_items)
    A = np.empty(p.shape, dtype="<U1")
    for j in range(n_items):
        wrong = [o for o in options if o != key[j]]
        w = rng.dirichlet(np.ones(len(wrong)))                          # mỗi câu một kiểu phương án nhiễu
        A[:, j] = np.where(correct[:, j], key[j], rng.choice(wrong, size=n_persons, p=w))
    items = [f"q{j + 1:03d}" for j in range(n_items)]
    users = [f"s{i + 1:04d}" for i in range(n_persons)]
    return (pd.DataFrame(A, index=users, columns=items),
            pd.Series(key, index=items),
            pd.DataFrame({"a": a, "b": b}, index=items))
 
 
A_sim, key_sim, truth = simulate_exam()
res_ctt = analyze(A_sim, key_sim)
print("KR-20:", round(res_ctt["kr20"], 3))
print(res_ctt["items"]["class"].value_counts())
chk = res_ctt["items"].join(truth)
print(chk[["p", "r_pbis", "a", "b"]].corr().round(2))                   # kỳ vọng: p~b rất âm, r_pbis~a dương
A_sim.to_csv(OUT / "exam_answers.csv")
key_sim.to_csv(OUT / "exam_key.csv", header=["key"])
res_ctt["items"].to_csv(OUT / "exam_ctt_items.csv")
res_ctt["distractors"].to_csv(OUT / "exam_distractors.csv", index=False)

KR-20: 0.928
class
phu hop                    84
qua de                     28
phan cach thap              4
qua kho, phan cach thap     4
Name: count, dtype: int64
           p  r_pbis     a     b
p       1.00    0.52  0.03 -0.98
r_pbis  0.52    1.00  0.62 -0.52
a       0.03    0.62  1.00 -0.01
b      -0.98   -0.52 -0.01  1.00
